In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 Johnny Nuñez Cano
# SPDX-License-Identifier: MIT
# Adapted from https://github.com/johnnynunez/blogs/tree/main/Article_2
# See ../LICENSE and ../THIRD_PARTY_NOTICES.md for attribution.


# Scaling with MuJoCo Warp


[MuJoCo Warp (MJWarp)](https://github.com/google-deepmind/mujoco_warp) implements MuJoCo physics using [NVIDIA Warp](https://github.com/NVIDIA/warp). It is developed jointly by Google DeepMind and NVIDIA for parallel simulation on NVIDIA GPUs.

Reuse the CPU lesson's scene and controller, then compare task behavior and measure batched physics throughput. The compiled SO-101 model retains the same **±30 N·m arm actuator override** and **0.002 s timestep**. Matching these settings is necessary for comparison, but differences in precision and solver implementation can still change contact-rich trajectories.


## Outline

- Distinguish single-step latency from aggregate world-step throughput.
- Refactor the CPU task to use `mujoco_warp.step` with explicit device state.
- Inspect the resulting task behavior against the CPU reference.
- Measure fixed-control physics throughput over increasing batch sizes.

---

## What you will learn

- Select a CPU or GPU execution path based on measured workload needs.
- Upload a model with `put_model` and allocate batched state with `make_data`.
- Set contact and constraint capacities with `nconmax` and `njmax`.
- Use CUDA graph capture to reduce repeated kernel-launch overhead.


## Introduction to Warp and MJWarp

**Warp** compiles typed Python kernel functions for execution on CPUs or CUDA GPUs. A `@wp.kernel` function operates on Warp arrays; the selected device determines where those arrays and computations live. This tutorial uses Warp arrays through MJWarp.

**MJWarp** maps supported MuJoCo model features onto batched device data and operations such as `mjw.step` and `mjw.forward`. Read the [MJWarp overview](https://mujoco.readthedocs.io/en/stable/mjwarp/index.html) for its latency/throughput tradeoff and supported use cases.

| Workload | Execution path to evaluate |
|----------|-----------------------------|
| Many independent simulations alongside a GPU policy | MJWarp batched GPU simulation |
| Interactive simulation or a latency-sensitive individual step | MuJoCo CPU |
| JAX transformations and multi-device execution | MuJoCo XLA (MJX) |
| CPU batching across multiple cores | `mujoco.rollout` |

A GPU step can take longer than an individual CPU step while advancing many more worlds. Whether that improves an application depends on batch size, contact workload, hardware, and transfers between host and device.


## Setup

In a local checkout, install the packages using the [tutorial README](../README.md) and select that environment as the notebook kernel. The cell below locates the checkout and changes into this lesson's directory so that the exercise scripts and imports resolve correctly.

In Google Colab, the cell downloads this tutorial from the repository's `main` branch and installs its pinned dependencies. For the GPU lesson, select **Runtime → Change runtime type → T4 GPU** (or another available NVIDIA GPU) first. If Colab requests a runtime restart after installation, restart and rerun the setup cell.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

lesson_relative = Path("tutorials/warp/notebooks/mujoco/part2")
repo_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / lesson_relative / "pick_place_common.py").is_file()),
    None,
)
if repo_root is None and in_colab:
    repo_root = Path("/content/accelerated-computing-hub")
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
             "--branch", "main",
             "https://github.com/NVIDIA/accelerated-computing-hub.git",
             str(repo_root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(repo_root), "sparse-checkout", "set",
             "tutorials/warp/notebooks/mujoco"],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
if repo_root is None or not (repo_root / lesson_relative / "pick_place_common.py").is_file():
    raise FileNotFoundError("Open this notebook from an accelerated-computing-hub checkout.")

if in_colab:
    # Colab has no desktop display; use its GPU-backed EGL context.
    os.environ.setdefault("MUJOCO_GL", "egl")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(repo_root / "tutorials/warp/notebooks/mujoco/requirements.txt")],
        check=True,
    )
lesson_dir = repo_root / lesson_relative
os.chdir(lesson_dir)
if str(lesson_dir) not in sys.path:
    sys.path.insert(0, str(lesson_dir))
print("Lesson directory:", lesson_dir)


## Environment check

This lesson's throughput measurements require a CUDA GPU. The completed GPU scripts require CUDA and stop if no CUDA device is available. Keep the pinned dependency set in `requirements.txt` together when reproducing this tutorial.


In [ ]:
!nvidia-smi

In [ ]:
import importlib.metadata
import warp as wp
import mujoco_warp as mjw
import mujoco

wp.init()
device = wp.get_device()

print("warp        :", importlib.metadata.version("warp-lang"))
print("mujoco-warp :", importlib.metadata.version("mujoco-warp"))
print("mujoco      :", mujoco.__version__)
print("device      :", device)
print("is CUDA     :", device.is_cuda)
print("all devices :", [str(d) for d in wp.get_devices()])
if not device.is_cuda:
    raise RuntimeError("Select an NVIDIA CUDA GPU runtime before continuing this lesson.")


## Overview of the files

| File | Purpose |
|------|---------|
| `so101_mjwarp.py` | Exercise script for the migration |
| `scaling_study.py` | Completed tool for measuring fixed-control physics throughput |
| `robots.py`, `pick_place_common.py`, `utils.py` | The same task helpers used in the CPU lesson |
| `solutions/` | Completed script and one snippet per exercise step |

The shared helpers preserve the scene, initialization, and controller while the execution path changes. Compare them directly:


In [ ]:
assert Path("../part1/pick_place_common.py").read_bytes() == Path("pick_place_common.py").read_bytes()
print("The CPU and GPU lessons use the same task helper.")


## Modify the simulation script

`so101_mjwarp.py` contains two modes and eight exercises.

**Task demonstration** (`run_demo`) simulates one world and copies its state back to a CPU `MjData` for visualization and task validation. Use it to inspect whether the migrated task behaves as intended.

**Physics benchmark** (`benchmark`) initializes many independent states from the same home pose, holds the controls fixed, and measures batched physics steps. It excludes the waypoint/IK controller, host transfers, rendering, resets, and task-success checks from the timed region. It does not measure a batch of successful pick-and-place tasks.

Back up the exercise before editing it. The completed reference is `solutions/so101_mjwarp_solution.py`.


In [ ]:
import shutil

backup = Path("so101_mjwarp_original.py")
if not backup.exists():
    shutil.copyfile("so101_mjwarp.py", backup)
print("Backup:", backup)


### 0. Import MJWarp and Warp

At `TODO Step 0`, add:

```python
import mujoco_warp as mjw
import warp as wp
```

Keep `mujoco`: the standard bindings compile the MJCF model and run the CPU-side IK used in the task demonstration.


### 1. Upload the model to the GPU

At `TODO Step 1`, replace `m = None` with `mjw.put_model(mjm)`.

This creates an `mjw.Model` with fields stored in Warp arrays. Configure the model before upload; this example shares that configured model across its world states. Consult the [API compatibility table](https://github.com/google-deepmind/mujoco_warp#mujoco-api-compatibility) for unsupported features and implementation differences.


### 2. Allocate batched device state

At `TODO Step 2`, call `mjw.make_data(mjm, nworld=1, nconmax=nconmax, njmax=njmax)`.

| Parameter | Meaning |
|-----------|---------|
| `nworld` | Number of worlds. State arrays such as `qpos` have shape `(nworld, nq)`. |
| `nconmax` | Contact capacity averaged across worlds; the total budget is `nworld * nconmax`. A world can use more than its share if the total fits. |
| `njmax` | Constraint capacity for each world. |

Use one world for the task demonstration. Increase the count later for the physics benchmark. These capacities are passed to `make_data`; the deprecated MJCF `size` attributes do not configure them.


### 3. Seed the device state

At `TODO Step 3`, copy the host state onto the device and run a forward pass.

Two conversions happen on every copy:

- **Shape**: `mjd.qpos` has shape `(nq,)` but `d.qpos` has shape `(nworld, nq)`, so index with `[None, :]` to add the leading world axis.
- **Precision**: MJWarp is `float32`; MuJoCo defaults to double. Expect small numerical differences, and be careful with tight friction or convergence-sensitive setups.

Copy `qpos`, `qvel` and `ctrl`, then call `mjw.forward(m, d)` — the MJWarp equivalent of `mj_forward`, which recomputes derived quantities without advancing time.

### 4. Push the control vector to the device

At `TODO Step 4`, copy `mjd.ctrl` into `d.ctrl` with `wp.copy`, using `[None, :]` to add the world dimension.

This demonstration transfers controls from the CPU on every substep. That makes the data movement easy to inspect, but repeated transfers and synchronization add overhead. A production GPU policy loop would normally keep more of this computation on the device.


### 5. Step the physics on the GPU

At `TODO Step 5`, call `mjw.step(m, d)`.

One call advances every world in the batch by one timestep. Step time can grow more slowly than batch size while the GPU has spare capacity, but neither constant step time nor a particular speedup is guaranteed. Measure the curve for this scene and device.


### 6. Mirror world 0 back to the host

At `TODO Step 6`, copy `qpos` and `qvel` back for the host viewer and task checks:

```python
mjd.qpos[:] = d.qpos.numpy()[0]
mjd.qvel[:] = d.qvel.numpy()[0]
```

`.numpy()` copies the Warp array back to host memory, and `[0]` selects world 0 out of the batch. Be aware that this is a synchronisation point: it blocks until the GPU finishes. In a real training loop you would keep observations on device and feed them straight to the policy.

### 7. Capture the step into a CUDA graph

At `TODO Step 7`, record the operations for a physics step:

```python
graph = None
if device.is_cuda:
    with wp.ScopedCapture() as capture:
        mjw.step(m, d)
    graph = capture.graph
```

Replaying a CUDA graph with `wp.capture_launch(graph)` reduces repeated host submission overhead for the step's collection of kernels. It does not remove the cost of computing physics.

- Initialize state and configure model options before capture.
- Replays use captured buffers; update their contents rather than replacing array objects.
- Capture and compilation are outside the timed region. Warm up the execution path and synchronize the device before starting and after ending the measurement.
- CUDA graphs require a CUDA device. Both this lesson and its completed GPU scripts require CUDA; do not label another execution device as GPU performance.


## Validate task behavior against the CPU reference

Run the completed exercise headless and inspect final cube positions, `xy_err`, and `dz`. A successful stack should have a small horizontal error and a vertical center separation near 0.044 m. Compare the trajectory and task outcome as well as these final values.

The reference script's `--test` option checks finite state, overflow/warnings, controller completion, and the settled stack. This is a validation exercise, not an assertion that the CPU and GPU versions already match. This tutorial uses `float64` MuJoCo CPU state and `float32` MJWarp state; numerical and contact-solver differences can change grasp success. Recheck model options, timestep, contact capacities, and controller timing if behavior differs.


In [ ]:
!{sys.executable} so101_mjwarp.py --headless-steps 600
# !{sys.executable} so101_mjwarp.py --robot rebot --headless-steps 600
# Automated acceptance checks in the completed reference:
!{sys.executable} solutions/so101_mjwarp_solution.py --headless-steps 600 --test


Contact or constraint overflow means the allocated capacities are insufficient. Increase `--nconmax` or `--njmax` as appropriate and rerun the complete task, including its most demanding contact phases. Do not interpret a fast run with overflow as a valid benchmark.


## Measure fixed-control physics throughput

After completing the exercises, run the benchmark below. It times 200 batch steps after warm-up at a 0.002 s timestep, with fixed home-pose controls. Start with a smaller `--nworld` if device memory is limited.

**World-steps per second = measured batch steps × number of worlds ÷ measured wall-clock seconds.** This excludes setup, compilation, controller execution, rendering, and task-success evaluation. To run without completing the exercises, use `solutions/so101_mjwarp_solution.py` with the same arguments.


In [ ]:
!{sys.executable} so101_mjwarp.py --benchmark --nworld 4096 --steps 200

Sweep the batch size using the completed scaling tool. Compare `ms/step` (batch-step latency) with `world-steps/s` (aggregate throughput). Use the same measured step count as the CPU baseline:


In [ ]:
!{sys.executable} scaling_study.py --steps 200


Throughput rises when adding worlds uses GPU capacity more efficiently; it can plateau or fall as memory and compute limits are reached. The curve is an observation to make, not a shape to assume.

The `speedup` column compares aggregate throughput with the first MJWarp batch in this sweep; it is **not a CPU/GPU speedup**. Compare against the fixed-control CPU measurement separately, and include a threaded CPU rollout baseline if that is a realistic deployment option. The CPU and GPU versions use different precision, so report that alongside the matched timestep and task configuration.


## Reproducibility and scope

Record package versions, device, world count, timestep, contact capacities, solver settings, graph usage, warm-up, and timed steps. Repeat measurements; short timings can be noisy.

Matching initial states does not imply bitwise deterministic GPU trajectories. Use numerical tolerances and task-level checks rather than exact float equality when comparing runs. Changing packages, precision, or solver settings requires revalidating both behavior and throughput.

These results describe a compact manipulator scene under fixed controls. A different robot, more contacts, a learned policy, rendering, or resets can change the bottleneck. Consult the [official performance guidance](https://mujoco.readthedocs.io/en/stable/mjwarp/index.html#performance-tuning) before generalizing to another workload.


In [ ]:
print("Keep the package-version and device output above with your results.")
print("Report timestep=0.002 s and fixed home-pose controls.")
print("Record nworld, contact capacities, measured steps, and graph usage from the run.")


## Verify against the reference

Compare your file against the reference implementation:

In [ ]:
!diff so101_mjwarp.py solutions/so101_mjwarp_solution.py
# The reference also adds validation helpers and an import path for solutions/.
# Focus on the completed TODO steps; the files need not be byte-identical.


## Recap

- How can a longer batch step still deliver more world-steps per second?
- What happens when contact or constraint capacities are too small?
- Which setup and compilation costs are excluded from the measurement?
- Why does the task demonstration transfer state while the benchmark does not?
- Why does fixed-control physics throughput not measure successful manipulation tasks per second?

---

## Next

Vary one benchmark parameter at a time and record the resulting curve. Keep the CPU task as a behavior reference, and add application-specific validation before using a new execution path for learning or control. Return to the [tutorial README](../README.md) for runnable reference commands and dependencies.
